# Extraction Results — Exact Match Evaluation

Micro-averaged Precision, Recall, F1 per category and overall.

**Micro-averaging** pools raw triple counts (matched, predicted, gold) before dividing — the correct approach when entries contain very few triples (1–3) and categories have different sizes.

In [95]:
import json
import pandas as pd
from collections import defaultdict
import re

RESULTS_PATH = "results/results_gemini_gemini-3-flash-preview_rdf_1.jsonl"

# ── Load results ──────────────────────────────────────────────────────────────
records = []
with open(RESULTS_PATH, encoding="utf-8") as f:
    for line in f:
        records.append(json.loads(line))

df = pd.DataFrame(records)
df["category"] = df["entry_id"].str.rsplit("_", n=2).str[0]
df["n_gold"] = df["gold_triples"].apply(len)
df["n_pred"] = df["pred_triples"].apply(len)

def add_underscores(text):
    """Replace any non-alphanumeric symbols that separate words with underscores, keep numbers."""
    return re.sub(r'[^a-zA-Z0-9]+', '_', text).strip('_')

def process_triple(triple):
    """Add underscores to subject and object in a triple."""
    return {
        'subject': add_underscores(triple['subject']),
        'relation': triple['relation'],
        'object': add_underscores(triple['object'])
    }

# Apply to all gold triples
df['gold_triples'] = df['gold_triples'].apply(
    lambda triples: [process_triple(t) for t in triples]
)

# Reapply to predicted triples to use updated function
df['pred_triples'] = df['pred_triples'].apply(
    lambda triples: [process_triple(t) for t in triples]
)


print(f"Loaded {len(df)} entries across {df['category'].nunique()} categories")
print(f"Gold triples: {df['n_gold'].sum()}  |  Predicted triples: {df['n_pred'].sum()}")
df[["entry_id", "category", "n_gold", "n_pred"]].head(10)

Loaded 990 entries across 57 categories
Gold triples: 1919  |  Predicted triples: 1886


,entry_id,category,n_gold,n_pred
0,1_Airport_dev_1,1_Airport,1,1
1,1_Airport_dev_2,1_Airport,1,1
2,1_Airport_dev_3,1_Airport,1,1
3,1_Airport_dev_4,1_Airport,1,1
4,1_Airport_dev_5,1_Airport,1,1
5,1_Airport_dev_6,1_Airport,1,1
6,1_Airport_dev_7,1_Airport,1,1
7,1_Airport_dev_8,1_Airport,1,1
8,1_Airport_dev_9,1_Airport,1,1
9,1_Airport_dev_10,1_Airport,1,1


In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

FUZZY_THRESHOLD = 0.85
SYNONYM_GROUPS  = [
    {"currentTeam", "formerTeam", "club", "currentClub"},
    {"country", "isPartOf"},
]

# Load once — reused for all similarity calls
print("Loading embedding model...")
embedder = SentenceTransformer("all-MiniLM-L6-v2")
print("Done.")

# ── helpers ───────────────────────────────────────────────────────────────────
def exact_match(pred: dict, gold: dict) -> bool:
    return (pred["subject"].lower()  == gold["subject"].lower()  and
            pred["relation"].lower() == gold["relation"].lower() and
            pred["object"].lower()   == gold["object"].lower())

def relations_are_synonyms(r1: str, r2: str) -> bool:
    r1, r2 = r1.lower(), r2.lower()
    return any(r1 in g and r2 in g for g in
               [{x.lower() for x in grp} for grp in SYNONYM_GROUPS])

def cosine(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-10))

def fuzzy_match(pred: dict, gold: dict,
                emb_cache: dict, threshold: float = FUZZY_THRESHOLD) -> bool:
    """Relation must match exactly; subject & object matched by embedding similarity."""
    if pred["relation"].lower() != gold["relation"].lower():
        return False
    for key in ("subject", "object"):
        pv, gv = pred[key].lower(), gold[key].lower()
        if pv == gv:
            continue
        if pv not in emb_cache:
            emb_cache[pv] = embedder.encode(pv, convert_to_numpy=True)
        if gv not in emb_cache:
            emb_cache[gv] = embedder.encode(gv, convert_to_numpy=True)
        if cosine(emb_cache[pv], emb_cache[gv]) < threshold:
            return False
    return True

def synonym_match(pred: dict, gold: dict,
                  emb_cache: dict, threshold: float = FUZZY_THRESHOLD) -> bool:
    """Like fuzzy_match but also accepts synonym relations."""
    rel_ok = (pred["relation"].lower() == gold["relation"].lower() or
              relations_are_synonyms(pred["relation"], gold["relation"]))
    if not rel_ok:
        return False
    for key in ("subject", "object"):
        pv, gv = pred[key].lower(), gold[key].lower()
        if pv == gv:
            continue
        if pv not in emb_cache:
            emb_cache[pv] = embedder.encode(pv, convert_to_numpy=True)
        if gv not in emb_cache:
            emb_cache[gv] = embedder.encode(gv, convert_to_numpy=True)
        if cosine(emb_cache[pv], emb_cache[gv]) < threshold:
            return False
    return True

def count_matched_full(pred_triples: list, gold_triples: list,
                       emb_cache: dict):
    """
    Returns (exact, fuzzy_extra, synonym_extra, fuzzy_corrections, synonym_corrections).
    fuzzy_corrections  — list of (pred, gold) pairs fixed by fuzzy but not exact.
    synonym_corrections — list of (pred, gold) pairs fixed by synonym but not fuzzy.
    No double-counting: each gold triple matched at most once (best match wins).
    """
    used_exact = set(); used_fuzzy = set(); used_syn = set()
    exact_n = fuzzy_n = syn_n = 0
    fuzzy_corrections = []
    synonym_corrections = []

    # exact pass
    for p in pred_triples:
        for j, g in enumerate(gold_triples):
            if j not in used_exact and exact_match(p, g):
                exact_n += 1
                used_exact.add(j)
                break

    # fuzzy pass (only on gold not already matched exactly)
    used_fuzzy = set(used_exact)
    for p in pred_triples:
        # skip if this pred was already counted in exact
        already = any(j in used_exact and exact_match(p, gold_triples[j])
                      for j in used_exact)
        if already:
            continue
        for j, g in enumerate(gold_triples):
            if j not in used_fuzzy and fuzzy_match(p, g, emb_cache):
                fuzzy_n += 1
                used_fuzzy.add(j)
                fuzzy_corrections.append({"pred": p, "gold": g})
                break

    # synonym pass (only on gold not already matched by exact or fuzzy)
    used_syn = set(used_fuzzy)
    for p in pred_triples:
        already_exact = any(j in used_exact and exact_match(p, gold_triples[j])
                            for j in used_exact)
        already_fuzzy = any(c["pred"] == p for c in fuzzy_corrections)
        if already_exact or already_fuzzy:
            continue
        for j, g in enumerate(gold_triples):
            if j not in used_syn and synonym_match(p, g, emb_cache):
                syn_n += 1
                used_syn.add(j)
                synonym_corrections.append({"pred": p, "gold": g})
                break

    return exact_n, fuzzy_n, syn_n, fuzzy_corrections, synonym_corrections

# ── run over full dataframe ───────────────────────────────────────────────────
print("Computing matches (embedding calls may take a moment)...")
emb_cache = {}

results = df.apply(
    lambda r: count_matched_full(r["pred_triples"], r["gold_triples"], emb_cache),
    axis=1
)

df["matched_exact"]        = results.apply(lambda x: x[0])
df["matched_fuzzy_extra"]  = results.apply(lambda x: x[1])
df["matched_syn_extra"]    = results.apply(lambda x: x[2])
df["fuzzy_corrections"]    = results.apply(lambda x: x[3])
df["synonym_corrections"]  = results.apply(lambda x: x[4])

# cumulative matched columns
df["matched_fuzzy"]    = df["matched_exact"] + df["matched_fuzzy_extra"]
df["matched_synonym"]  = df["matched_fuzzy"] + df["matched_syn_extra"]

print(f"Exact matched   : {df['matched_exact'].sum()}")
print(f"+ Fuzzy extra   : {df['matched_fuzzy_extra'].sum()}  → {df['matched_fuzzy'].sum()} total")
print(f"+ Synonym extra : {df['matched_syn_extra'].sum()}  → {df['matched_synonym'].sum()} total")

Loading embedding model...
Done.
Computing matches (embedding calls may take a moment)...
Exact matched   : 1473
+ Fuzzy extra   : 64  → 1537 total
+ Synonym extra : 28  → 1565 total


In [97]:
def micro_prf(group, matched_col):
    m       = group[matched_col].sum()
    p_total = group["n_pred"].sum()
    g_total = group["n_gold"].sum()
    prec = m / p_total if p_total else 0.0
    rec  = m / g_total if g_total else 0.0
    f1   = 2*prec*rec / (prec+rec) if (prec+rec) else 0.0
    return pd.Series({"Precision": prec, "Recall": rec, "F1": f1,
                      "Matched": int(m), "Gold": int(g_total),
                      "Predicted": int(p_total), "Entries": len(group)})

cat_exact   = df.groupby("category").apply(micro_prf, matched_col="matched_exact",   include_groups=False)
cat_fuzzy   = df.groupby("category").apply(micro_prf, matched_col="matched_fuzzy",   include_groups=False)
cat_synonym = df.groupby("category").apply(micro_prf, matched_col="matched_synonym", include_groups=False)

# side-by-side F1
comparison = pd.DataFrame({
    "F1_exact":   cat_exact["F1"],
    "F1_fuzzy":   cat_fuzzy["F1"],
    "F1_synonym": cat_synonym["F1"],
    "Δ_fuzzy":    cat_fuzzy["F1"]   - cat_exact["F1"],
    "Δ_synonym":  cat_synonym["F1"] - cat_fuzzy["F1"],
    "Gold":       cat_exact["Gold"],
    "Entries":    cat_exact["Entries"],
}).sort_index()

comparison.style.format({c: "{:.3f}" for c in comparison.columns if c not in ("Gold","Entries")})

,F1_exact,F1_fuzzy,F1_synonym,Δ_fuzzy,Δ_synonym,Gold,Entries
category,,,,,,,
1_Airport,0.886,0.886,0.886,0.000,0.000,35.000000,35.000000
1_Artist,0.938,0.938,0.938,0.000,0.000,32.000000,32.000000
1_Astronaut,0.625,0.625,0.625,0.000,0.000,8.000000,8.000000
1_Athlete,0.844,0.875,0.875,0.031,0.000,32.000000,32.000000
1_Building,0.880,0.880,0.880,0.000,0.000,25.000000,25.000000
1_CelestialBody,0.947,0.947,0.947,0.000,0.000,19.000000,19.000000
1_City,0.481,0.593,0.593,0.111,0.000,27.000000,27.000000
1_ComicsCharacter,1.000,1.000,1.000,0.000,0.000,11.000000,11.000000
1_Company,0.667,0.857,0.857,0.190,0.000,10.000000,10.000000


In [98]:
def overall_prf(matched_col):
    return micro_prf(df, matched_col)

ov_exact   = overall_prf("matched_exact")
ov_fuzzy   = overall_prf("matched_fuzzy")
ov_synonym = overall_prf("matched_synonym")

for label, ov in [("EXACT", ov_exact), ("FUZZY (≥0.9 sim)", ov_fuzzy), ("SYNONYM-AWARE", ov_synonym)]:
    print(f"\n{'─'*50}")
    print(f"{label}")
    print(f"  Precision : {ov['Precision']:.4f}")
    print(f"  Recall    : {ov['Recall']:.4f}")
    print(f"  F1        : {ov['F1']:.4f}")
    print(f"  Matched   : {int(ov['Matched'])} / {int(ov['Gold'])} gold, {int(ov['Predicted'])} pred")

print(f"\n  Fuzzy  fixed {int(ov_fuzzy['Matched']  - ov_exact['Matched'])} extra triples "
      f"(F1 +{ov_fuzzy['F1']-ov_exact['F1']:.4f})")
print(f"  Synonym fixed {int(ov_synonym['Matched'] - ov_fuzzy['Matched'])} extra triples "
      f"(F1 +{ov_synonym['F1']-ov_fuzzy['F1']:.4f})")


──────────────────────────────────────────────────
EXACT
  Precision : 0.7810
  Recall    : 0.7676
  F1        : 0.7742
  Matched   : 1473 / 1919 gold, 1886 pred

──────────────────────────────────────────────────
FUZZY (≥0.9 sim)
  Precision : 0.8150
  Recall    : 0.8009
  F1        : 0.8079
  Matched   : 1537 / 1919 gold, 1886 pred

──────────────────────────────────────────────────
SYNONYM-AWARE
  Precision : 0.8298
  Recall    : 0.8155
  F1        : 0.8226
  Matched   : 1565 / 1919 gold, 1886 pred

  Fuzzy  fixed 64 extra triples (F1 +0.0336)
  Synonym fixed 28 extra triples (F1 +0.0147)


In [99]:
fuzzy_rows = df[df["matched_fuzzy_extra"] > 0][["entry_id","category","input_text","fuzzy_corrections"]]

for _, row in fuzzy_rows.iterrows():
    print(f"\n{'='*80}")
    print(f"{row['entry_id']}  [{row['category']}]")
    print(f"Text: {row['input_text'][:120]}")
    for c in row["fuzzy_corrections"]:
        print(f"  PRED : ({c['pred']['subject']}, {c['pred']['relation']}, {c['pred']['object']})")
        print(f"  GOLD : ({c['gold']['subject']}, {c['gold']['relation']}, {c['gold']['object']})")
        print()


1_Athlete_dev_32  [1_Athlete]
Text: The owner of the Atlanta Falcons is Arthur Blank.
  PRED : (the_Atlanta_Falcons, owner, Arthur_Blank)
  GOLD : (Atlanta_Falcons, owner, Arthur_Blank)


1_City_dev_12  [1_City]
Text: The area of water, in Anaheim (California), is 25.2 square kilometres.
  PRED : (Anaheim_California, areaOfWater, 25_2)
  GOLD : (Anaheim, areaOfWater, 25_2)


1_City_dev_13  [1_City]
Text: Auburn (Alabama) is part of Alabama.
  PRED : (Auburn_Alabama, isPartOf, Alabama)
  GOLD : (Auburn, isPartOf, Alabama)


1_City_dev_25  [1_City]
Text: Madison County (Indiana) is in the United States.
  PRED : (Madison_County_Indiana, country, United_States)
  GOLD : (Madison_County, country, United_States)


1_Company_dev_3  [1_Company]
Text: Trane makes building management systems.
  PRED : (Trane, product, building_management_systems)
  GOLD : (Trane, product, Building_Management_System)


1_Company_dev_9  [1_Company]
Text: Digify, Inc. is a subsidiary of GMA New Media.
  PRED : (G

In [100]:
syn_rows = df[df["matched_syn_extra"] > 0][["entry_id","category","input_text","synonym_corrections"]]

for _, row in syn_rows.iterrows():
    print(f"\n{'='*80}")
    print(f"{row['entry_id']}  [{row['category']}]")
    print(f"Text: {row['input_text'][:120]}")
    for c in row["synonym_corrections"]:
        print(f"  PRED : ({c['pred']['subject']}, {c['pred']['relation']}, {c['pred']['object']})")
        print(f"  GOLD : ({c['gold']['subject']}, {c['gold']['relation']}, {c['gold']['object']})")
        print()


2_Airport_dev_19  [2_Airport]
Text: Al-Taqaddum Air Base serves the city of Fallujah which is in Iraq.
  PRED : (Fallujah, isPartOf, Iraq)
  GOLD : (Fallujah, country, Iraq)


2_Athlete_dev_14  [2_Athlete]
Text: Former clubs of Alessio Romagnoli include U.C. Sampdoria and A.S. Roma.
  PRED : (Alessio_Romagnoli, formerTeam, U_C_Sampdoria)
  GOLD : (Alessio_Romagnoli, club, U_C_Sampdoria)

  PRED : (Alessio_Romagnoli, formerTeam, A_S_Roma)
  GOLD : (Alessio_Romagnoli, club, A_S_Roma)


2_Building_dev_14  [2_Building]
Text: Amdavad ni Gufa is located in Ahmedabad in India.
  PRED : (Ahmedabad, isPartOf, India)
  GOLD : (Ahmedabad, country, India)


2_Building_dev_18  [2_Building]
Text: Amdavad ni Gufa is located in Gujarat, India.
  PRED : (Gujarat, isPartOf, India)
  GOLD : (Gujarat, country, India)


2_City_dev_4  [2_City]
Text: Akron is part of the United States, where Washington, D.C. is the capital.
  PRED : (Akron, country, United_States)
  GOLD : (Akron, isPartOf, United_States)



In [107]:
# uses matched_synonym as the final "correct" count
errors_inds = df[df["matched_synonym"] != df["n_gold"]].index.tolist()

def index_generator():
    for ind in errors_inds:
        yield ind

gen = index_generator()

In [124]:
idx = next(gen)
row = df.iloc[idx]

print("=" * 80)
print(f"Index: {idx} | Entry ID: {row['entry_id']}")
print(f"Category: {row['category']} | "
      f"Exact: {row['matched_exact']}/{row['n_gold']} | "
      f"Fuzzy: {row['matched_fuzzy']}/{row['n_gold']} | "
      f"Synonym: {row['matched_synonym']}/{row['n_gold']}")
print("-" * 80)
print(f"Input: {row['input_text']}")
print("-" * 80)
print("GOLD TRIPLES:")
for i, t in enumerate(row["gold_triples"], 1):
    print(f"  {i}. ({t['subject']}, {t['relation']}, {t['object']})")
print("-" * 80)
print("PRED TRIPLES:")
for i, t in enumerate(row["pred_triples"], 1):
    print(f"  {i}. ({t['subject']}, {t['relation']}, {t['object']})")
print("=" * 80)

Index: 119 | Entry ID: 1_Building_dev_13
Category: 1_Building | Exact: 0/1 | Fuzzy: 0/1 | Synonym: 0/1
--------------------------------------------------------------------------------
Input: Adisham Hall has the Tudor Revival architectural style.
--------------------------------------------------------------------------------
GOLD TRIPLES:
  1. (Adisham_Hall, architecturalStyle, Tudor_Revival_architecture)
--------------------------------------------------------------------------------
PRED TRIPLES:
  1. (Adisham_Hall, architecturalStyle, Tudor_Revival)


In [125]:
def calculate_embedding_similarity(text1: str, text2: str) -> float:
    """Calculate cosine similarity between embeddings of two texts."""
    emb1 = embedder.encode(text1, convert_to_numpy=True)
    emb2 = embedder.encode(text2, convert_to_numpy=True)
    return cosine(emb1, emb2)

similarity = calculate_embedding_similarity("Tudor_Revival", "Tudor_Revival_architecture")
print(f"Similarity: {similarity:.4f}")

Similarity: 0.7963


In [115]:
row = df.iloc[67]

print("=" * 80)
print(f"Index: {idx} | Entry ID: {row['entry_id']}")
print(f"Category: {row['category']} | "
      f"Exact: {row['matched_exact']}/{row['n_gold']} | "
      f"Fuzzy: {row['matched_fuzzy']}/{row['n_gold']} | "
      f"Synonym: {row['matched_synonym']}/{row['n_gold']}")
print("-" * 80)
print(f"Input: {row['input_text']}")
print("-" * 80)
print("GOLD TRIPLES:")
for i, t in enumerate(row["gold_triples"], 1):
    print(f"  {i}. ({t['subject']}, {t['relation']}, {t['object']})")
print("-" * 80)
print("PRED TRIPLES:")
for i, t in enumerate(row["pred_triples"], 1):
    print(f"  {i}. ({t['subject']}, {t['relation']}, {t['object']})")
print("=" * 80)

Index: 67 | Entry ID: 1_Astronaut_dev_1
Category: 1_Astronaut | Exact: 0/1 | Fuzzy: 0/1 | Synonym: 0/1
--------------------------------------------------------------------------------
Input: Alan Shepard was a crew member of Apollo 14.
--------------------------------------------------------------------------------
GOLD TRIPLES:
  1. (Alan_Shepard, mission, Apollo_14)
--------------------------------------------------------------------------------
PRED TRIPLES:
  1. (Apollo_14, crewMembers, Alan_Shepard)
